# Notebook 3 — Model Training & Backtesting

Train the LSTM model, evaluate it on the test set, run a backtest, and plot the equity curve.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import DataFetcher
from src.data.processor import DataProcessor
from src.models.lstm_model import LSTMModel
from src.training.trainer import Trainer
from src.training.evaluator import Evaluator
from src.backtesting.backtest import Backtester
from src.utils.config import load_config

config = load_config()
config['data']['source'] = 'yfinance'
config['data']['timeframe'] = '1Day'
config['training']['epochs'] = 50  # Reduce for notebook speed; increase for better results

SYMBOL = 'AAPL'

In [ ]:
# 1. Fetch & process data
fetcher = DataFetcher(config)
processor = DataProcessor(config)

raw_df = fetcher.fetch(SYMBOL, start='2020-01-01', end='2024-12-31')
feat_df = processor.process(raw_df)

# Update input_size to actual feature count
config['model']['input_size'] = feat_df.shape[1]
print(f'Input features: {feat_df.shape[1]}')

In [ ]:
# 2. Build sequences and split
X, y = processor.build_sequences(feat_df, target_col='close')
X_train, y_train, X_val, y_val, X_test, y_test = processor.train_val_test_split(X, y)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

In [ ]:
# 3. Build model and train
model = LSTMModel.from_config(config)
trainer = Trainer(model, config)

history = trainer.train(X_train, y_train, X_val, y_val, symbol=SYMBOL)

In [ ]:
# Plot training curves
fig = go.Figure()
fig.add_trace(go.Scatter(y=history['train_loss'], name='Train Loss'))
fig.add_trace(go.Scatter(y=history['val_loss'], name='Val Loss'))
fig.update_layout(title='Training & Validation Loss', xaxis_title='Epoch', yaxis_title='MSE Loss')
fig.show()

In [ ]:
# 4. Load best checkpoint and evaluate on test set
trainer.load_best(symbol=SYMBOL)
evaluator = Evaluator(model, trainer.device)

print('\n── Test Set Metrics ──')
metrics = evaluator.evaluate(X_test, y_test)

# Get predictions for plotting
y_pred = evaluator.predict(X_test)

In [ ]:
# Plot predicted vs actual (normalized scale)
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test, name='Actual', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=y_pred, name='Predicted', line=dict(color='orange', dash='dash')))
fig.update_layout(title=f'{SYMBOL} — Predicted vs Actual (Test Set, Normalized)',
                  xaxis_title='Time Step', yaxis_title='Normalized Close')
fig.show()

In [ ]:
# 5. Inverse transform predictions back to USD
actual_prices = processor.inverse_transform_close(y_test, list(feat_df.columns))
pred_prices = processor.inverse_transform_close(y_pred, list(feat_df.columns))

fig = go.Figure()
fig.add_trace(go.Scatter(y=actual_prices, name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=pred_prices, name='Predicted Price', line=dict(color='orange', dash='dash')))
fig.update_layout(title=f'{SYMBOL} — Predicted vs Actual Close Price (USD)',
                  xaxis_title='Time Step', yaxis_title='Close Price ($)')
fig.show()

In [ ]:
# 6. Backtest
backtester = Backtester(config)
summary, equity_df = backtester.run(actual_prices, pred_prices, threshold=0.002)

print('\n── Backtest Summary ──')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# Plot equity curve
fig = go.Figure()
fig.add_trace(go.Scatter(x=equity_df['step'], y=equity_df['equity'],
                         name='Portfolio Equity', fill='tozeroy',
                         line=dict(color='green')))
fig.add_hline(y=config['backtesting']['initial_capital'],
              line_dash='dash', line_color='gray', annotation_text='Starting Capital')
fig.update_layout(title=f'{SYMBOL} — Equity Curve (Backtest)',
                  xaxis_title='Time Step', yaxis_title='Portfolio Value ($)')
fig.show()